# 📊 Notebook 15: Fallstudie – E-Commerce Analyse

**Kurs:** Datenaufbereitung und -verarbeitung (DAV) · THWS BBA

---

## Szenario

Du analysierst die Bestelldaten eines deutschen Online-Shops. Drei Tabellen liegen getrennt vor:
- **Bestellungen**: 3.000 Bestellungen (2023-2024)
- **Kunden**: 800 Kundendatensätze
- **Produkte**: 50 Produktkatalogeinträge

**Deine Aufgaben:**
- Drei Tabellen zusammenführen (3-Tabellen-Join)
- Datenqualität prüfen und bereinigen
- Umsatz-Pivot nach Land und Kategorie
- Retourenquote analysieren
- RFM-Analyse vorbereiten

**DAV-Techniken:** Merging · Pivot-Tabellen · RFM-Analyse · Plotly

In [ ]:
# ── Setup: Pakete installieren (nur in Colab nötig) ──────────────────────────
import subprocess, sys
packages = ['ydata-profiling', 'missingno', 'plotly', 'kaleido']
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)
print('Setup abgeschlossen!')

In [ ]:
# ── Imports & Daten-Basis-URL ─────────────────────────────────────────────────
BASE_URL = "https://raw.githubusercontent.com/swrobuts/dav/main/data/"

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')
print('Bibliotheken geladen!')

## 1. Datenmodell verstehen

Drei Tabellen mit folgenden Beziehungen:

```
ecommerce_bestellungen.csv
  bestellnummer | kunden_id → ecommerce_kunden.csv
               | produkt_id → ecommerce_produkte.csv
```

Wir laden zunächst alle drei Tabellen separat.

In [ ]:
# Alle drei Tabellen laden
bestellungen = pd.read_csv(BASE_URL + "ecommerce_bestellungen.csv")
kunden = pd.read_csv(BASE_URL + "ecommerce_kunden.csv")
produkte = pd.read_csv(BASE_URL + "ecommerce_produkte.csv")

print(f"Bestellungen: {bestellungen.shape}")
print(f"Kunden: {kunden.shape}")
print(f"Produkte: {produkte.shape}")

print("\nBestellungen (Vorschau):")
bestellungen.head(3)

In [ ]:
# Jede Tabelle kurz inspizieren
print("=== Bestellungen ===")
print(bestellungen.dtypes)

print("\n=== Kunden ===")
print(kunden.dtypes)

print("\n=== Produkte ===")
print(produkte)


## 2. Datenqualität prüfen

In [ ]:
# Fehlende Werte in allen drei Tabellen
print("Fehlende Werte – Bestellungen:")
print(bestellungen.isnull().sum()[bestellungen.isnull().sum() > 0])

print("\nFehlende Werte – Kunden:")
print(kunden.isnull().sum()[kunden.isnull().sum() > 0])

print("\nFehlende Werte – Produkte:")
print(produkte.isnull().sum()[produkte.isnull().sum() > 0])

# Bewertungen: Viele NaN erwartet (nicht alle Kunden bewerten)
bewertet = bestellungen['bewertung'].notna().sum()
print(f"\nBestellungen mit Bewertung: {bewertet} ({bewertet/len(bestellungen)*100:.1f}%)")

## 3. Drei-Tabellen-Join

Wir verbinden alle drei Tabellen zu einem vollständigen Datensatz.

In [ ]:
# Schritt 1: Bestellungen + Kunden
df = bestellungen.merge(kunden, on='kunden_id', how='left')
print(f"Nach Join 1 (Bestellungen + Kunden): {df.shape}")

# Schritt 2: + Produkte
df = df.merge(produkte, on='produkt_id', how='left')
print(f"Nach Join 2 (+ Produkte): {df.shape}")

# Datum konvertieren
df['datum'] = pd.to_datetime(df['datum'])

print("\nSpalten im joined Datensatz:")
print(list(df.columns))
df.head(3)

In [ ]:
# Fehlende Bewertungen: Median pro Kategorie imputen
median_by_cat = df.groupby('kategorie_y')['bewertung'].transform('median')
df['bewertung'] = df['bewertung'].fillna(median_by_cat).round()

print(f"Fehlende Bewertungen nach Imputation: {df['bewertung'].isna().sum()}")
print(f"Bewertungs-Verteilung:\n{df['bewertung'].value_counts().sort_index()}")

## 4. Umsatz-Pivot: Land × Kategorie

Wir erstellen eine Pivot-Tabelle, die zeigt, welche Kategorie in welchem Land am meisten Umsatz macht.

In [ ]:
# Pivot: Umsatz pro Land x Kategorie
pivot_umsatz = df.pivot_table(
    values='preis_eur',
    index='land',
    columns='kategorie_y',
    aggfunc='sum',
    fill_value=0
).round(0)

print("Umsatz-Pivot (EUR):")
print(pivot_umsatz)

fig = px.imshow(
    pivot_umsatz,
    title='Umsatz (EUR) nach Land × Produktkategorie',
    color_continuous_scale='Blues',
    aspect='auto',
    text_auto='.0f'
)
fig.update_layout(xaxis_title='Kategorie', yaxis_title='Land')
fig.show()

In [ ]:
# Retourenquote nach Land und Kategorie
retouren_land = df.groupby('land')['retourniert'].mean().reset_index()
retouren_land.columns = ['land', 'retourenquote']
retouren_land['retourenquote_pct'] = (retouren_land['retourenquote'] * 100).round(1)
retouren_land = retouren_land.sort_values('retourenquote', ascending=False)

fig = px.bar(
    retouren_land, x='land', y='retourenquote_pct',
    title='Retourenquote nach Land (%)',
    color='retourenquote_pct',
    color_continuous_scale='Reds',
    text='retourenquote_pct'
)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.show()

## 5. RFM-Analyse

**RFM** misst Kundenwert anhand von:
- **R**ecency: Wie lange ist der letzte Kauf her?
- **F**requency: Wie oft wurde gekauft?
- **M**onetary: Wie viel wurde ausgegeben?

Kunden mit niedrigem R, hohem F und hohem M sind die wertvollsten Kunden.

In [ ]:
# Nur erfolgreiche Bestellungen (nicht retourniert)
df_erfolg = df[df['retourniert'] == False].copy()

# Referenzdatum = letzter Tag im Datensatz
today = df_erfolg['datum'].max()

rfm = df_erfolg.groupby('kunden_id').agg(
    Recency=('datum', lambda x: (today - x.max()).days),
    Frequency=('bestellnummer', 'count'),
    Monetary=('preis_eur', 'sum')
).reset_index()

rfm['Monetary'] = rfm['Monetary'].round(2)

print(f"RFM-Analyse für {len(rfm)} Kunden:")
print(rfm.describe())
rfm.head(10)

In [ ]:
# RFM-Scatter: Frequency vs. Monetary (Größe = Recency invertiert)
rfm['Recency_inv'] = rfm['Recency'].max() - rfm['Recency'] + 1

fig = px.scatter(
    rfm,
    x='Frequency',
    y='Monetary',
    size='Recency_inv',
    color='Recency',
    color_continuous_scale='RdYlGn_r',
    title='RFM-Analyse: Frequency vs. Monetary (Größe = Aktualität)',
    hover_data=['kunden_id', 'Recency'],
    labels={'Frequency': 'Anzahl Bestellungen', 'Monetary': 'Gesamtumsatz (EUR)', 'Recency': 'Tage seit letztem Kauf'}
)
fig.show()

# Top 10 wertvollste Kunden
top_kunden = rfm.sort_values(['Frequency', 'Monetary'], ascending=[False, False]).head(10)
print("\nTop 10 wertvollste Kunden:")
print(top_kunden[['kunden_id', 'Recency', 'Frequency', 'Monetary']].to_string(index=False))

## 6. Umsatz-Zeitreihe

In [ ]:
# Monatlicher Umsatz
df['monat'] = df['datum'].dt.to_period('M').astype(str)
monthly_revenue = df[df['retourniert'] == False].groupby('monat')['preis_eur'].sum().reset_index()
monthly_revenue.columns = ['monat', 'umsatz']

fig = px.line(
    monthly_revenue, x='monat', y='umsatz',
    title='Monatlicher Umsatz (EUR, ohne Retouren)',
    markers=True,
    color_discrete_sequence=['#0389CF']
)
fig.update_layout(xaxis_title='Monat', yaxis_title='Umsatz (EUR)')
fig.update_xaxes(tickangle=45)
fig.show()

## 7. Zusammenfassung & Erkenntnisse

### Was haben wir gelernt?

**DAV-Techniken angewendet:**
- **3-Tabellen-Join** mit `merge()` (left join für vollständige Bestellhistorie)
- **Fehlende Werte** bei Bewertungen mit Median-Imputation bereinigt
- **Pivot-Tabellen** für multidimensionale Umsatzanalyse
- **RFM-Analyse** für Kundensegmentierung
- **Zeitreihen-Aggregation** für Monatsumsatz

**Erkenntnisse:**
- Elektronik und Mode dominieren den Umsatz in allen Ländern
- Retourenquote liegt bei ~15% (branchenüblich)
- Wertvollste Kunden kaufen häufig und haben hohen Gesamtumsatz
- Saisonales Muster: Q4 zeigt typischerweise höhere Umsätze (Weihnachten)

**Nächste Schritte (für ML):**
- RFM-Scores für Clustering berechnen
- Retourenvorhersage-Modell entwickeln
- Customer Lifetime Value (CLV) berechnen